# Programming exercise 10: Ground State Search with the Variational Monte Carlo Method

Due on Monday, 29.06.2026, 20h



In [ ]:

import flax.linen as nn

import jax
import jax.numpy as jnp

import numpy as np

import os

import scipy.sparse as sparse 
import scipy.sparse.linalg as sLA

import optax

import matplotlib.pyplot as plt

import Comp_Quant_Dynam as cqd

# JAX wants to work on GPUs by default, 
# which we have to explicitly bypass if no GPUs are available on device
os.environ["JAX_PLATFORM_NAME"] = "cpu"

We will again be looking at the transverse field Ising model (TFIM) in 1D from exercise sheet 8,
$$
H=\sum_{i=0}^{N-1} -J\sigma_z^{(i)}\sigma_z^{(i+1)} - B \sigma_x^{(i)}.
$$
In this sheet we will construct two different variational ansatze: A Jastrow ansatz and a Feed-Forward Neural Network ansatz in order to do a ground state search by minimizing the variational energy,

$$E_\text{var}(\mathbf{\theta}) = \frac{\braket{\psi_\mathbf{\theta}|H|\psi_\mathbf{\theta}}}{\braket{\psi_\mathbf{\theta}|\psi_\mathbf{\theta}}} \geq E_\text{gs}.$$

Thus, the ansatze will be representing the variational wave function $\braket{\mathbf{s}|\psi_\mathbf{\theta}}=\psi_{\mathbf{\theta}}(\mathbf{s})$ and we will optimize the variational parameters $\mathbf{\theta}$, such that after convergence $\ket{\psi_\mathbf{\theta}}$ corresponds to the ground state of the TFIM Hamiltonian. We will use our results from sheet 8 to benchmark the convergence of our algorithm.

We will be using some libraries that are state-of-the-art for optimization tasks with neural networks such as `jax` (https://docs.jax.dev/en/latest/index.html), `flax` (https://flax-linen.readthedocs.io/en/latest/#) and `optax` (https://optax.readthedocs.io/en/latest/). 

### Brief introduction to the libraries: `jax`, `flax` and `optax`

`jax` is a "[...] is a library for array-oriented numerical computation (à la NumPy), with automatic differentiation and JIT compilation to enable high-performance machine learning research" developed by Google. `flax` is the `jax`-based library for constructing artificial neural networks and `optax` is the `jax`-based library of optimizers for training neural networks, all in a (possibly GPU) accelarated way.


Additionally to auto-diff and JIT compilation via Open XLA, JAX functions can be automatically vectorized and are compatible with fast parallel execution on GPUs. All numpy functions that you will need are copied from numpy and optimized by jax under the package `jax.numpy` 

In [ ]:
# One can define arrays in the same way as with numpy
sx = jnp.array([[0,1],[1,0]])/2

# and use matrix operations
sx2 = jnp.dot(sx, sx)
matexp = jnp.exp(sx2)

print(matexp)

For variational (Monte Carlo) optimization tasks, one usually needs to use random sampling with the help of pseudo-random-number generator (PRNG) keys. Every key takes a seed such that the result is pseudo-random yet reproducible. JAX has many tools developed in `jax.random` precisely for this purpose.

In [ ]:
seed = 2 # Choose a random seed 
key = jax.random.PRNGKey(seed) # Define a PRNG key to use 

num_samples = 200

x_samples = jax.random.normal(key, shape=(num_samples)) # Sample from a normal distribution

def normal(x):
    return jnp.sqrt(1/(2 * jnp.pi)) * jnp.exp(-x**2/2)
x = jnp.linspace(-3, 3, 100)

plt.plot(x, normal(x), label="normal distr.")
plt.scatter(x_samples, jnp.zeros_like(x_samples), label="samples", alpha=0.2)
plt.legend()

One side-effect of JAX' XLA optimized code is that one has to avoid using regular `if`, `for` and `while` loops for accelerated computing on GPUs, since these can make the otherwise highly optimized code either crash or largely slow down. Here, we will be running code firstly on our local computer, thus you can use regular loop structures. But for making use of the GPU accelaration, one uses the `jax`-internal alternatives from `jax.lax`. For example a for-loop can be written using `jax.lax.scan`

In [ ]:
def step_function(carry, x):
    next_carry = carry + x
    return next_carry, next_carry


numbers = jnp.array([1, 2, 3, 4, 5])
initial_sum = 0

final_sum, running_sums = jax.lax.scan(step_function, initial_sum, numbers)

print("Final state:", final_sum)
print("History of states:", running_sums)

### Defining variational models via `flax`
Using `flax.linen` we can create neural networks or other variational ansatze in a simple class definition. 

In [ ]:
class myModel(nn.Module):
        
        # We define our class as an `nn.Module` s.t. it inherits the flax.linen structure

        """  """

        out_dim: int = 1
        actfunc: callable = nn.tanh # Choose a suitable activation function

        @nn.compact
        def __call__(self, x):     

            # Here we define how inputs x are called and passed through the network
            
            out = self.actfunc(nn.Dense(self.out_dim, kernel_init=jax.nn.initializers.normal(),
                          bias_init=jax.nn.initializers.zeros)(x))   # We initialize the w
            

            return out[..., 0]
        

# Initialize the class
mymodel = myModel(out_dim=1, actfunc=nn.tanh)


dummy_input = jnp.ones((1, 10))
key = jax.random.PRNGKey(2)        

# Initialize the model with random parameters using a dummy input        
params = mymodel.init(key, dummy_input)

# Notice that the parameters of a flax module are stored as a dictionary NOT an array
print("Data type of the flax params:", type(params))

# We can take a closer look at the params structure 
print("Initialized Parameters Structure:",jax.tree_util.tree_map(lambda x: x.shape, params))

# If one wishes one can ravel the parameters dictionary into an array 
params_flat, unravel_func = jax.flatten_util.ravel_pytree(params)
print("Model parameters as a flattened array:", params_flat)

# The unravel function then knows how to wrap the aray back into the right dictionary form
# In order to apply the parameters, they have to be in the dictionary form
params =  unravel_func(params_flat)

# We can compute outputs of our model by applying the model onto inputs
output = mymodel.apply(params, dummy_input)
print(rf"Output of the randomly initialized model: {output}") 

### Exercise 1: The Jastrow Ansatz and Markov Chain Monte Carlo (MCMC)

We will start by constructing a short-range Jastrow ansatz that entangles nearest and next-to nearest neigbours,

$$\braket{\mathbf{s}|\psi_\text{jas}} = \exp\left(\sum\limits_{i}J_1 \sigma^z_i\sigma^z_{i+1} + J_2 \sigma^z_i\sigma^z_{i+2}\right),$$

where the parameters $J_1$ and $J_2$ have to be learned. We will construct the Jastrow ansatz using a `flax` based class. It is common to define the logarithm of the variational wave functions so we define `Jastrow()`$=\log \psi_\text{jas}$

- After defining the Jastrow ansatz, initialize the model and test the output on a spin state $\ket{111...111}$ with $N=10$. 

In order to do ground state search via variational Monte Carlo, we need a sampling algorithm that can generate random vectors of $N$ spins. But not all spin configurations are equally likely! That's why we use Markov Chain Monte Carlo (MCMC) to sample from the Born distribution of the variational wave function $p_{\mathbf{\theta}}(\mathbf{s}) = \frac{|{\psi_{\mathbf{\theta}}(\mathbf{s})}|^2}{\braket{\psi_{\mathbf{\theta}}|\psi_{\mathbf{\theta}}}}$. 

For this, we have implemented a sampler via the Metropolis-Hastings algorithm using `jax.lax.scan`. Make sure you understand, how the sampler works. Compare the code to the algorithm sketch below: At each step
- Start with configuration s
- Propose a new spin state s' by doing a random spin flip (0 -> 1 or 1 -> 0)
- Accept the proposed new spin state with acceptance probability

$$p_\text{accept}(s,s') = \min \left( 1 , \frac{p_{\mathbf{\theta}}(\mathbf{s'})}{p_{\mathbf{\theta}}(\mathbf{s})}\right)$$
- Otherwise keep s
- Repeat until $N_\text{samp}$ samples have been accepted.

Why is there a nested "full sweep" loop within the outer loop? Why is this important? Think about the independency and autocorrelations of samples. 

In [ ]:
num_samples = 15
N_spins = 10


spin_samples = cqd.utility.MCMC_Sampler_Metropolis_Hastings(model=mymodel,
                                                            params=params, 
                                                            init_state = jnp.ones((N_spins,)), 
                                                            num_samples=num_samples,
                                                            PRNGkey= jax.random.PRNGKey(7))


# Use the dummy model from above to generate some random spin samples
print(f"Let us look at {num_samples} spin state samples:")
print(" ")
print(spin_samples)

### Exercise 2: Ground State Search via Stochastic Gradient Descent (SGD) 

Before we define our ground state search algorithm, let us define our minimization objective, the variational energy expectation value $E_\text{var}(\mathbf{\theta})$ which is given by,

$$
E_\text{var}(\mathbf{\theta}) =  \sum\limits_{s} \frac{|\psi_{\mathbf{\theta}}(s)|^2}{\braket{\psi_{\mathbf{\theta}}|\psi_{\mathbf{\theta}}}} E_\text{loc} (s) = \langle E_\text{loc} (s)\rangle_{s\sim p_{\mathbf{\theta}}},
$$
here $E_\text{loc} (s)$ is the local estimate of the energy that can be further written down as

$$
E_\text{loc} (s) = \frac{\bra{s}\hat{H}\ket{\psi_{\mathbf{\theta}}}}{\braket{s|\psi_{\mathbf{\theta}}}} = \sum\limits_{s'} \bra{s}\hat{H}\ket{s'} \frac{\psi_{\mathbf{\theta}}(s')}{\psi_{\mathbf{\theta}}(s)}.
$$

Now, we can implement the gradient of the variational energy $E_\text{var}(\mathbf{\theta})$ as the Monte Carlo averaged expression,

$$\nabla_{\theta_k}E_\text{var}(\mathbf{\theta}) = 2 \, \text{Re} \left[ \langle O^*_k(s) E_\text{loc} (s)\rangle_{s\sim p_{\mathbf{\theta}}} - \langle O^*_k(s)\rangle_{s\sim p_{\mathbf{\theta}}} \langle E_\text{loc} (s)\rangle_{s\sim p_{\mathbf{\theta}}} \right],$$

with the logarithmic variational derivatives $O_k (s) = \partial_{\theta_k} \log \psi_{\mathbf{\theta}}(s)$ and update the variational parameters accordingly via gradient descent,

$$\theta^{(i+1)} = \theta^{(i)} - \eta \nabla_{\theta_k}E_\text{var}(\mathbf{\theta}), $$

using the optax optimizer `optax.adam(learning_rate=lr)` or  `optax.adabelief(learning_rate=lr)` and the update functions `optimizer.update()` as well as `optax.apply_updates()`.

- Plot the variational ground state energy $E_\text{gs}(\mathbf{\theta})$ against the number of iterations and show its convergence by comparing to the exact ground state energy
- (Optional): Play around with different learning rates $\eta_i$ and number of MC samples $N_\text{MC}$ to see the effect on the speed of convergence and statistical fluctuations

### Exercise 3: Ground State Search using Neural Quantum States

Now we can build the Neural Quantum States (NQS) ansatz (https://www.science.org/doi/10.1126/science.aag2302) . Construct a feed-forward neural network (FFNN), with randomly initialized network parameters. For spin systems usually, the FFNN takes a vector of spins $\mathbf{\sigma}$ as inputs and return the logarithm of the spin wave function as an output, 

$$\log\psi_{\mathbf{\theta}}(\mathbf{\sigma}) = \frac{1}{2}\log \rho_{\mathbf{\theta}}(\mathbf{\sigma}) + i \phi_{\mathbf{\theta}}(\mathbf{\sigma}).$$

But here the ground state wave function of the 1D TFIM is purely real, so we can construct a FFNN model that outputs a single real number.

Choose a suited nonlinear activation function, and initialize the FFNN with random variational parameters $\mathbf{\theta}$ chosen from a normal distribution. The $l$-th layer of the Feed-Forward Neural Network consists of the weight matrix $W^{(l)}_{kl}$ and bias vector $b^{(l)}_l$ and the output is sent through a nonlinear activation function,

$$\text{FFNN}^{(l+1)}(\mathbf{s}) = \text{ActFunc}\left( \sum\limits_{k}W^{(l)}_{kl}s^{(l)}_k + b^{(l)}_l\right).$$

Initialize the weights $W^{(l)}_{kl}$ from a normal distribution and the biases $b^{(l)}_l$ from zero. `jax.nn.initializers.lecun_normal` and `jax.nn.initializers.zeros` functions may be helpful.

- Run the ground search algorithm now for the NQS ansatz and compare the convergence in a plot with the Jastrow ansatz using again the analytical ground state energy benchmark.
- How does the NQS ansatz perform for large and small network sizes and larger and smaller learning rates? What are potential reasons for this performance and possible ways to improve?

(Optional): In order to get an intuition of how the structure of the FFNN changes its performance you can play around with different hidden layer numbers and activation functions on https://playground.tensorflow.org, especially for regression problems.

### Exercise 4: Ground state of the tilted TFIM 

In this exercise we look at the tilted 1D TFIM,

$$
H=\sum_{i=0}^{N-1} -J\sigma_z^{(i)}\sigma_z^{(i+1)} - B \sigma_x^{(i)} - g \sigma_z^{(i)}.
$$

where we add a longitudinal field $g$ to the system which breaks the spin-flip symmetry. Thus, for this system there exists no analytical solution of the ground state energy and exact diagonalization is only possible to a certain number of spins. Here comes the advantage of variational methods truly into play. Define the Hamiltonian from above analogously to sheet 8 and solve the ground state energy using exact diagonalization roughly for spin numbers $N\in \{2,\dots, 20\}$ and find the same ground state via variational Monte Carlo with your model of choice. Then push the limits of what is possible with ED by using the variational model for finding the ground state energies for larger systems $N\in \{21,\dots, 100\}$, as long as your computer can handle it.


### (Optional) Exercise 5: Testing the GPU Acceleration

Now, we want to test the true GPU acceleration that working with `jax` provides us. Therefore, go to Google Collab (https://colab.research.google.com), where you can upload this python notebook. Go to the upper right corner ("Connect") and choose to connect with device: T4 GPU. You need to first install the CQD package locally on to the cluster via:

`!pip install git+https://github.com/NiklasEuler/CQD_SS26.git `

Now, make sure that the line `os.environ["JAX_PLATFORM_NAME"] = "cpu"` is commented out and `jax` should automatically communicate with the GPU device, which is referred to as `CUDA device`.

- Make sure, that both the MCMC sampler and the ground state search functions are defined purely with `jax.lax` loops such as `jax.lax.scan` or `jax.lax.fori_loop` and that trivially parallelizable batch evaluations are vectorized via `jax.vmap`. And enforce `JIT` compilation by using `@jax.jit` infront of your loss function.
- Run the ground state search from above on the GPUs and compare the runtime difference. 
- Now that the code runs faster, you might want to do a ground state search for even higher number of spins. 

In [ ]:
# Test how many GPU devices are visible
print("Available devices:", jax.devices())